# Module 1: Scraping Setup and Execution 

### 📌 Step 1: Import Required Libraries                                                                                                   
🔹 Objective:                                                                                                                      
Import all necessary Python libraries required for web scraping and data handling.

In [3]:
pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\PADMINI\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from tqdm import tqdm
import time

### 📌 Step 2: Scrape Power Index Ranking                                                                                              
🔹 Objective:                                                                                                                      
Extract:                                                                                                                            
- Country Name                                                                                                                        
- Power Index Rank                                                                                                                    
- Power Index Score                                                                                                                   
from:                 
👉 https://www.globalfirepower.com/countries-listing.php                                                                           
🔹 Function to Scrape Power Index

In [30]:
def scrape_power_index():
    url = "https://www.globalfirepower.com/countries-listing.php"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", class_="recordsetContainer")

    data = []

    for i, card in enumerate(cards):
        rank = i + 1  # Rank from position

        country_tag = card.find("div", class_="longFormName")
        if not country_tag:
            continue

        country = country_tag.get_text(strip=True)

        full_text = card.get_text(" ", strip=True)
        score_match = re.search(r"\d+\.\d+", full_text)
        score = float(score_match.group()) if score_match else None

        data.append([country, rank, score])

    return pd.DataFrame(
        data,
        columns=["country", "power_index_rank", "power_index_score"]
    )

🔹 Execute the Function

In [8]:
power_df = scrape_power_index()
power_df.head()

,country,power_index_rank,power_index_score
0,United States,1,0.0741
1,Russia,2,0.0791
2,China,3,0.0919
3,India,4,0.1346
4,South Korea,5,0.1642


### 📌 Step 3: Create Number Extraction Helper Function                                                                                
🔹 Objective: Extract numeric values from scraped text and remove commas.

In [16]:
def extract_number(text):
    if pd.isna(text):
        return None

    text = str(text).replace(",", "")
    match = re.search(r"[\d.]+", text)
    return float(match.group()) if match else None

### 📌 Step 4: Create Generic Metric Scraper Function                                                                                  
🔹 Objective:
Create a reusable function to scrape different metric pages.

In [12]:
def scrape_metric_page(url, metric_name):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", class_="recordsetContainer")

    data = []

    for card in cards:
        country_tag = card.find("div", class_="longFormName")
        if not country_tag:
            continue

        country = country_tag.get_text(strip=True)

        value_tag = card.find("div", class_="valueContainer")
        if not value_tag:
            continue

        raw_text = value_tag.get_text(" ", strip=True)
        value = extract_number(raw_text)

        data.append([country, value])

    return pd.DataFrame(data, columns=["country", metric_name])

### 📌 Step 5: Define All Metric URLs                                                                                                  
🔹 Objective:
Store all metric URLs and their corresponding column names in a dictionary.

In [17]:
master_df = power_df.copy()
other_sources = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
        'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
        'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
        'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
        'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
        'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
        'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
        'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
        'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
        'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
        'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
        'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
        'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
        'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
        'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
        'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
        'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
        'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
        'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
        'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
        'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
        'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
        'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
        'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
        'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
        'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
        'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
        'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
        'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
        'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
        'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
        'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
        'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
        'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
        'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
        'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
        'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
        'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
        'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
        'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
        'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
        'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
        'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
        'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
        'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
        'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
        'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
        'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
        'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
        'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
        'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
        'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
        'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
        'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}

### 📌 Step 6: Scrape All Metrics and Merge                                                                                            
🔹 Objective:
Loop through all metric pages and merge them with the main dataframe.

In [18]:
for url, metric_name in tqdm(other_sources.items()):
    df_metric = scrape_metric_page(url, metric_name)

    master_df = pd.merge(
        master_df,
        df_metric,
        on="country",
        how="outer"
    )

    time.sleep(2)  # polite delay to avoid blocking

100%|██████████████████████████████████████| 54/54 [02:42<00:00,  3.00s/it]


### 📌 Step 7: Export Final Dataset                                                                                                    
- Objective:
Save the complete dataset to CSV.

In [19]:
master_df.to_csv("military_raw_data.csv", index=False)

To print the first five rows in the dataset

In [20]:
master_df.head()

,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,Afghanistan,121,2.7342,40121552.0,15647405.0,8826741.0,842553.0,75000.0,0.0,90000.0,...,8.020000e+07,8.020000e+07,4.955400e+10,767000.0,503000.0,66000000.0,652230.0,0.0,5987.0,1200.0
1,Albania,77,1.7258,3107100.0,1522479.0,1292554.0,62142.0,7500.0,0.0,500.0,...,5.062300e+07,5.062300e+07,5.692000e+09,473000.0,255000.0,522000000.0,28748.0,362.0,691.0,41.0
2,Algeria,27,0.4849,47022473.0,22570787.0,19185169.0,752360.0,130000.0,150000.0,150000.0,...,1.007260e+11,4.796300e+10,4.504000e+12,0.0,3000.0,233000000.0,2381740.0,998.0,6734.0,0.0
3,Angola,59,1.1045,37202061.0,7440412.0,3720206.0,372021.0,107000.0,0.0,10000.0,...,5.514000e+09,1.397000e+09,3.430020e+11,0.0,0.0,0.0,1246700.0,1600.0,5369.0,1300.0
4,Argentina,32,0.5983,46994384.0,20677529.0,17575900.0,704916.0,108000.0,12370.0,20000.0,...,4.328000e+10,4.622800e+10,3.964640e+11,869000.0,2534000.0,799999000.0,2780400.0,4989.0,11968.0,11000.0


.shape function tells you the size (dimensions) of your DataFrame.

In [21]:
master_df.shape

(145, 57)

To Print Column Names Line by Line

In [29]:
for col in master_df.columns:
    print(col)

country
power_index_rank
power_index_score
total_population
total_military_manpower
fit_for_service
population_reaching_military_age_annually
active_personnel
reserve_personnel
paramilitary
total_military_aircraft
fighter_aircraft
attack_aircraft
transport_aircraft
trainer_aircraft
special_mission_aircraft
tanker_aircraft
total_military_helicopters
attack_helicopters
tanks
armored_fighting_vehicles
self_propelled_artillery
towed_artillery
rocket_projectors
total_naval_fleet
total_naval_fleet_tonnage_mt
aircraft_carriers
helicopter_carriers
submarines
destroyers
frigates
corvettes
coastal_patrol_craft
mine_warfare_craft
defense_budget_usd
external_debt_usd
purchasing_power_parity_usd
foreign_exchange_and_gold_reserves_usd
total_serviceable_airports
labour_force
major_ports_and_terminals
total_merchant_marine_fleet
railway_coverage_km
roadway_coverage_km
oil_production_bbl
oil_consumption_bbl
proven_oil_reserves_bbl
natural_gas_production_cum
natural_gas_consumption_cum
proven_natural_ga

country
power_index_rank
power_index_score
total_population
total_military_manpower
fit_for_service
population_reaching_military_age_annually
active_personnel
reserve_personnel
paramilitary
total_military_aircraft
fighter_aircraft
attack_aircraft
transport_aircraft
trainer_aircraft
special_mission_aircraft
tanker_aircraft
total_military_helicopters
attack_helicopters
tanks
armored_fighting_vehicles
self_propelled_artillery
towed_artillery
rocket_projectors
total_naval_fleet
total_naval_fleet_tonnage_mt
aircraft_carriers
helicopter_carriers
submarines
destroyers
frigates
corvettes
coastal_patrol_craft
mine_warfare_craft
defense_budget_usd
external_debt_usd
purchasing_power_parity_usd
foreign_exchange_and_gold_reserves_usd
total_serviceable_airports
labour_force
major_ports_and_terminals
total_merchant_marine_fleet
railway_coverage_km
roadway_coverage_km
oil_production_bbl
oil_consumption_bbl
proven_oil_reserves_bbl
natural_gas_production_cum
natural_gas_consumption_cum
proven_natural_gas_reserves_cum
coal_production_cum
coal_consumption_mt
proven_coal_reserves_cum
total_land_area_sq_km
coastline_coverage_km
border_coverage_km
waterway_coverage_km

# Module 2: Data Cleaning and Structuring 

### 1️⃣ Import Libraries

- pandas is used for reading and manipulating datasets.
- numpy helps with numerical operations.

In [1]:
import pandas as pd
import numpy as np

### 2️⃣ Load Dataset

- read_csv() loads the raw dataset into a DataFrame.
- df.head() displays the first 5 rows to verify proper loading.

In [3]:
df = pd.read_csv("military_raw_data.csv")
df.head()

,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,Afghanistan,121,2.7342,40121552.0,15647405.0,8826741.0,842553.0,75000.0,0.0,90000.0,...,8.020000e+07,8.020000e+07,4.955400e+10,767000.0,503000.0,66000000.0,652230.0,0.0,5987.0,1200.0
1,Albania,77,1.7258,3107100.0,1522479.0,1292554.0,62142.0,7500.0,0.0,500.0,...,5.062300e+07,5.062300e+07,5.692000e+09,473000.0,255000.0,522000000.0,28748.0,362.0,691.0,41.0
2,Algeria,27,0.4849,47022473.0,22570787.0,19185169.0,752360.0,130000.0,150000.0,150000.0,...,1.007260e+11,4.796300e+10,4.504000e+12,0.0,3000.0,233000000.0,2381740.0,998.0,6734.0,0.0
3,Angola,59,1.1045,37202061.0,7440412.0,3720206.0,372021.0,107000.0,0.0,10000.0,...,5.514000e+09,1.397000e+09,3.430020e+11,0.0,0.0,0.0,1246700.0,1600.0,5369.0,1300.0
4,Argentina,32,0.5983,46994384.0,20677529.0,17575900.0,704916.0,108000.0,12370.0,20000.0,...,4.328000e+10,4.622800e+10,3.964640e+11,869000.0,2534000.0,799999000.0,2780400.0,4989.0,11968.0,11000.0


- Shows the number of rows and columns.
- Useful to understand dataset size.

In [5]:
df.shape

(145, 57)

Displays column names.\
Helps identify:
- Spaces
- Special characters
- Inconsistent naming <br>
This step prepares us for column standardization.

In [6]:
df.columns

Index(['country', 'power_index_rank', 'power_index_score', 'total_population',
       'total_military_manpower', 'fit_for_service',
       'population_reaching_military_age_annually', 'active_personnel',
       'reserve_personnel', 'paramilitary', 'total_military_aircraft',
       'fighter_aircraft', 'attack_aircraft', 'transport_aircraft',
       'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft',
       'total_military_helicopters', 'attack_helicopters', 'tanks',
       'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft',
       'defense_budget_usd', 'external_debt_usd',
       'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd',
       'total_serviceable_airports', 'labour_force',
    

3️⃣ Column Standardization

#### This step cleans column names to make them:
- Lowercase
- Underscore-separated
- Free from special symbols
#### Why this is important:
- Tableau prefers clean field names
- Python referencing becomes easier
- Prevents syntax errors

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("%", "percent", regex=False)
    .str.replace("+", "", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(".", "", regex=False)
)

df.columns

Index(['country', 'power_index_rank', 'power_index_score', 'total_population',
       'total_military_manpower', 'fit_for_service',
       'population_reaching_military_age_annually', 'active_personnel',
       'reserve_personnel', 'paramilitary', 'total_military_aircraft',
       'fighter_aircraft', 'attack_aircraft', 'transport_aircraft',
       'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft',
       'total_military_helicopters', 'attack_helicopters', 'tanks',
       'armored_fighting_vehicles', 'self_propelled_artillery',
       'towed_artillery', 'rocket_projectors', 'total_naval_fleet',
       'total_naval_fleet_tonnage_mt', 'aircraft_carriers',
       'helicopter_carriers', 'submarines', 'destroyers', 'frigates',
       'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft',
       'defense_budget_usd', 'external_debt_usd',
       'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd',
       'total_serviceable_airports', 'labour_force',
    

### 4️⃣ Text Cleaning

#### Purpose: Remove non-numeric symbols from values.\
What this removes:
- Commas (1,000 → 1000)
- Percent signs (45% → 45)
- Dollar signs ($5000 → 5000)
- Plus symbols (+200 → 200)
#### Why?
- Numeric conversion will fail if symbols exist.
- Cleaning ensures smooth conversion to integers/floats.
- We exclude "country" because it is categorical text.

In [8]:
numeric_cols = df.columns.drop("country")

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

df.head()

,country,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,Afghanistan,121,2.7342,40121552.0,15647405.0,8826741.0,842553.0,75000.0,0.0,90000.0,...,80200000.0,80200000.0,49554000000.0,767000.0,503000.0,66000000.0,652230.0,0.0,5987.0,1200.0
1,Albania,77,1.7258,3107100.0,1522479.0,1292554.0,62142.0,7500.0,0.0,500.0,...,50623000.0,50623000.0,5692000000.0,473000.0,255000.0,522000000.0,28748.0,362.0,691.0,41.0
2,Algeria,27,0.4849,47022473.0,22570787.0,19185169.0,752360.0,130000.0,150000.0,150000.0,...,100726000000.0,47963000000.0,4504000000000.0,0.0,3000.0,233000000.0,2381740.0,998.0,6734.0,0.0
3,Angola,59,1.1045,37202061.0,7440412.0,3720206.0,372021.0,107000.0,0.0,10000.0,...,5514000000.0,1397000000.0,343002000000.0,0.0,0.0,0.0,1246700.0,1600.0,5369.0,1300.0
4,Argentina,32,0.5983,46994384.0,20677529.0,17575900.0,704916.0,108000.0,12370.0,20000.0,...,43280000000.0,46228000000.0,396464000000.0,869000.0,2534000.0,799999000.0,2780400.0,4989.0,11968.0,11000.0


### 5️⃣ Numeric Conversion

Converts cleaned text columns into numeric format.\
errors="coerce" converts invalid entries to NaN.\
#### df.info() verifies:
- Data types
- Non-null counts\

After this step:\
All metric columns should be int64 or float64.\
Only country remains object type.

In [9]:
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 57 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    145 non-null    object 
 1   power_index_rank                           145 non-null    int64  
 2   power_index_score                          145 non-null    float64
 3   total_population                           145 non-null    float64
 4   total_military_manpower                    145 non-null    float64
 5   fit_for_service                            145 non-null    float64
 6   population_reaching_military_age_annually  145 non-null    float64
 7   active_personnel                           145 non-null    float64
 8   reserve_personnel                          145 non-null    float64
 9   paramilitary                               145 non-null    float64
 10  total_military_aircraft   

### 6️⃣ Missing Value Handling

Calculates percentage of missing values per column.
#### Helps identify:
- Columns with high missing data
- Columns needing cleaning
- Required for evaluation (< 2% overall missing).

In [10]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent.sort_values(ascending=False)

total_naval_fleet_tonnage_mt                 64.827586
country                                       0.000000
railway_coverage_km                           0.000000
corvettes                                     0.000000
coastal_patrol_craft                          0.000000
mine_warfare_craft                            0.000000
defense_budget_usd                            0.000000
external_debt_usd                             0.000000
purchasing_power_parity_usd                   0.000000
foreign_exchange_and_gold_reserves_usd        0.000000
total_serviceable_airports                    0.000000
labour_force                                  0.000000
major_ports_and_terminals                     0.000000
total_merchant_marine_fleet                   0.000000
roadway_coverage_km                           0.000000
destroyers                                    0.000000
oil_production_bbl                            0.000000
oil_consumption_bbl                           0.000000
proven_oil

### 7️⃣ Validation

- Confirms correct data types.
- All numeric columns should not be object type.
- Ensures dataset structure is clean.

In [11]:
df.dtypes

country                                       object
power_index_rank                               int64
power_index_score                            float64
total_population                             float64
total_military_manpower                      float64
fit_for_service                              float64
population_reaching_military_age_annually    float64
active_personnel                             float64
reserve_personnel                            float64
paramilitary                                 float64
total_military_aircraft                      float64
fighter_aircraft                             float64
attack_aircraft                              float64
transport_aircraft                           float64
trainer_aircraft                             float64
special_mission_aircraft                     float64
tanker_aircraft                              float64
total_military_helicopters                   float64
attack_helicopters                           f

The below cell shows a statistical summary:
- Mean
- Median
- Min
- Max
  
Helps detect:
- Abnormal values
- Outliers

In [13]:
df.describe()

,power_index_rank,power_index_score,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
count,145.000000,145.000000,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,145.000000,...,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,145.000000,145.000000,145.000000
mean,73.000000,1.642772,5.456547e+07,2.581125e+07,2.019565e+07,8.546933e+05,1.624485e+05,2.371013e+05,1.218919e+05,359.041379,...,2.792901e+10,2.768592e+10,1.377471e+12,6.465795e+07,6.465180e+07,8.000759e+09,9.144517e+05,4695.979310,3583.358621,4345.241379
std,42.001984,1.135182,1.702420e+08,8.583942e+07,6.928101e+07,2.667628e+06,2.967677e+05,6.517850e+05,6.095925e+05,1191.810935,...,1.060376e+11,9.339564e+10,5.483828e+12,4.168413e+08,4.450332e+08,3.209863e+10,2.178886e+06,18166.860635,3717.060550,11728.517195
min,1.000000,0.074100,3.640360e+05,8.372800e+04,5.023700e+04,1.820000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.190000e+02,0.000000,0.000000,0.000000
25%,37.000000,0.651700,5.650957e+06,2.392390e+06,1.912981e+06,7.199100e+04,2.250000e+04,0.000000e+00,2.000000e+03,25.000000,...,0.000000e+00,2.986000e+06,0.000000e+00,0.000000e+00,1.400000e+04,0.000000e+00,8.360000e+04,47.000000,1253.000000,0.000000
50%,73.000000,1.543200,1.469705e+07,6.351857e+06,4.096295e+06,2.048300e+05,5.100000e+04,2.010000e+04,1.250000e+04,99.000000,...,8.020000e+07,2.635000e+09,8.495000e+09,2.700000e+04,1.096000e+06,4.000000e+07,2.742000e+05,713.000000,2524.000000,825.000000
75%,109.000000,2.396900,3.879481e+07,1.794604e+07,1.403738e+07,6.250060e+05,1.600000e+05,1.685000e+05,5.500000e+04,271.000000,...,1.227000e+10,2.071900e+10,2.406930e+11,4.476000e+06,9.720000e+06,1.410000e+09,7.561020e+05,2640.000000,5002.000000,2800.000000
max,145.000000,5.799100,1.415043e+09,7.641234e+08,6.268642e+08,2.395518e+07,2.035000e+06,5.000000e+06,6.800000e+06,13032.000000,...,1.029000e+12,9.143010e+11,4.780500e+13,4.805000e+09,5.191000e+09,2.478830e+11,1.709824e+07,202080.000000,22457.000000,102000.000000


### 8️⃣ Export Clean Dataset

- Saves cleaned dataset.
- index=False prevents extra index column.

In [14]:
df.to_csv("military_cleaned.csv", index=False)